# 콩당콩당 CrossEncoder Reranker LoRA 파인튜닝 파이프라인

본 노트북은 RAG 1단계 검색 결과(Top-10/20)를 입력받아 질문 적합성 고해상도 재정렬을 수행하기 위해 `BAAI/bge-reranker-large` 기반 모델을 PEFT(LoRA) 기법으로 미세 조정하는 실습 코드입니다.

In [ ]:
# 1. 패키지 설치
!pip install -q transformers peft datasets sentencepiece accelerate

In [ ]:
# 2. 임포트 및 구성 로드
import os
import yaml
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType
from datasets import load_dataset

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Reranker configuration loaded:", config)

In [ ]:
# 3. 파인튜닝 데이터 전처리
# 데이터셋 포맷: {'query': '질문', 'passage': '초록/글', 'label': 1 (매칭) or 0 (미매칭)}
from datasets import Dataset
dummy_rerank = {
    "query": ["만성 신부전 칼륨 조절", "당뇨 미세알부민 검사", "만성 신부전 칼륨 조절"],
    "passage": [
        "신부전 환자는 칼륨 섭취를 차단해야 심장 마비를 막을 수 있습니다.",
        "매년 알부민 검사를 실시해 신장 손상을 추적합니다.",
        "칼륨과 신장은 무관하므로 마음껏 오렌지를 먹어도 괜찮습니다."
    ],
    "label": [1.0, 1.0, 0.0]  # 유사성 레이블 (1에 가까울수록 강한 적합)
}
dataset = Dataset.from_dict(dummy_rerank)
print("리랭킹 데이터 구조 준비 완료.")

In [ ]:
# 4. Reranker Base Model 로드 및 PEFT(LoRA) 레이어 주입
model_name = config['model']['base_model_name_or_path']
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)

peft_config = LoraConfig(
    r=config['lora']['r'],
    lora_alpha=config['lora']['lora_alpha'],
    lora_dropout=config['lora']['lora_dropout'],
    target_modules=config['lora']['target_modules'],
    bias="none",
    task_type=TaskType.SEQ_CLS
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

In [ ]:
# 5. 학습 정의 및 실행
output_dir = "./models/bge-reranker-lor"
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Reranker LoRA 모델 저장 완료: {output_dir}")